# Bài 2 — Prototype (Cách A): Concept-Augmented Motion Mapping trên Flora

Thay vì cộng điểm hậu kỳ sau DiT hay dùng GRL, **Cách A** đưa trực tiếp biểu diễn concept suy ra từ xương vào **phía Motion của Flora**:
1. **Stage 0 (~4 giây):** huấn luyện `ConceptEncoder + CLIPConceptHead` trên feature 256-D chỉ bằng Masked BCE (không `lambda_proto`, không `GRL`) → giữ `Unseen Concept AUC ~ 0.83–0.86+` (kể cả unseen-only concepts), sau đó đóng băng.
2. **Phase 1 & Phase 2 (Flora thuần):** ghép vector concept suy ra từ xương (`soft_proto` tái tạo từ từ điển CLIP hoặc `q_clip` liên tục) vào `motion_encoder/decoder` (`concat`) hoặc cộng phần dư vào `motion latent` (`latent_add`).

**Trước khi chạy**: commit và push `zeroshot/concept_map/` lên GitHub.

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess, time, re
import numpy as np, pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/BaoHuy/zeroshot'
FEAT_ARCHIVE = '/content/drive/MyDrive/BaoHuy/flora_pretrain.zip'
OUT_DIR = f'{DRIVE_DIR}/concept_map_step1'
EXTRACT = '/content/flora_features'
os.makedirs(OUT_DIR, exist_ok=True)

REPO = '/content/thesis'
if not os.path.exists(REPO):
    try:
        from google.colab import userdata
        tok = userdata.get('GH_TOKEN')
    except Exception:
        tok = None
    url = f'https://{tok}@github.com/NguyenBaoHuy05/thesis.git' if tok else 'https://github.com/NguyenBaoHuy05/thesis.git'
    subprocess.run(['git', 'clone', '-q', '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
!git -C {REPO} log -1 --format='%h %s'
subprocess.run(['git', '-C', REPO, 'rev-parse', 'HEAD'], check=True, stdout=open(f'{OUT_DIR}/thesis_commit.txt', 'w'))

FLORA = f'{REPO}/zeroshot/Flora'
CMAP = f'{REPO}/zeroshot/concept_map'
need_flora = ['nesy_common.py', 'kg_semantics.py', 'kg/primitive_v2.json', 'kg/kg_rules_v2.json', 'semantics/ntu60_llm.txt', 'semantics/ntu120_llm.txt']
need_cmap = ['model_cr.py', 'trainer_cr.py', 'run_cr.py']
missing = [f'Flora/{p}' for p in need_flora if not os.path.exists(f'{FLORA}/{p}')]
missing += [f'concept_map/{p}' for p in need_cmap if not os.path.exists(f'{CMAP}/{p}')]
assert not missing, f'Repo chưa có {missing}: commit và push zeroshot/concept_map trước'
print('repo OK')

In [ ]:
!pip -q install transformers pyyaml pandas

stub = f'{FLORA}/models/shiftgcn/Temporal_shift/cuda/shift_cuda.py'
if not os.path.exists(stub):
    with open(stub, 'w') as f:
        f.write('# stub: Shift-GCN CUDA op is never called when use_features=True\n')

!python -c "import sys; sys.path[:0]=['{FLORA}','{CMAP}']; from model_cr import ConceptAugmentedMappingNet; from trainer_cr import ConceptAugmentedTrainer; print('import OK')"

## Giải nén feature 256-D đã trích sẵn

In [ ]:
assert os.path.exists(FEAT_ARCHIVE), f'Không thấy {FEAT_ARCHIVE} trên Drive'
!ls -lh {FEAT_ARCHIVE}

FEAT_NAMES = ('star_features', 'synse_features')

def find_feat(name):
    hits = [h for h in glob.glob(f'{EXTRACT}/**/{name}', recursive=True) if '__MACOSX' not in h]
    return hits[0] if hits else None

if not any(find_feat(n) for n in FEAT_NAMES):
    os.makedirs(EXTRACT, exist_ok=True)
    t0 = time.time()
    subprocess.run(['unzip', '-q', '-o', FEAT_ARCHIVE, '-x', '__MACOSX/*', '*/._*', '._*', '-d', EXTRACT], check=True)
    print(f'giải nén xong trong {time.time() - t0:.0f}s')

assert any(find_feat(n) for n in FEAT_NAMES), f'Trong {FEAT_ARCHIVE} không có star_features hay synse_features'
os.makedirs(f'{FLORA}/skeleton_features', exist_ok=True)
for name in FEAT_NAMES:
    src_dir = find_feat(name)
    link = f'{FLORA}/skeleton_features/{name}'
    if src_dir is None:
        continue
    if os.path.islink(link):
        os.remove(link)
    os.symlink(src_dir, link)
    print(name, sorted(f for f in os.listdir(link) if not f.startswith('._')))

## Thử nghiệm Cách A (STAR features, 48/12 & 110/10, seed 1)

Chỉ chạy 4 biến thể đề xuất (mốc đối chiếu `flora_base` đã có: `63.87 / 77.83`, `Last10: 61.04 / 75.28`):
- `ca_concat_soft`: ghép `[ske_256 || 0.5 · soft_proto]` vào `motion_encoder/decoder`
- `ca_concat_q`: ghép `[ske_256 || 0.5 · q_clip]` vào `motion_encoder/decoder`
- `ca_lat_soft`: cộng phần dư `0.2 · soft_proto` vào `motion latent`
- `ca_lat_q`: cộng phần dư `0.2 · q_clip` vào `motion latent`

In [ ]:
!python {CMAP}/run_cr.py --feat star --tasks 48 110 --seeds 1 --variants ca_concat_soft ca_concat_q ca_lat_soft ca_lat_q --out-root {OUT_DIR} --force

## Xem chi tiết Concept Inference & Demo Top-5 concept (`ca_concat_soft`)

In [ ]:
for split_task in ('ntu60_xsub_seen48_unseen12', 'ntu120_xsub_seen110_unseen10'):
    print(f'=== {split_task} (ca_concat_soft) ===')
    log_path = f'{OUT_DIR}/star/ca_concat_soft/{split_task}_ZSL_seed1/run.log'
    if os.path.exists(log_path):
        for line in open(log_path, encoding='utf-8'):
            if '[CA-EVAL]' in line or '[CA-DEMO]' in line or 'Best ZSL' in line:
                print(line.strip())
    else:
        print('Chưa có log:', log_path)

## (Tùy chọn) Kiểm chứng thêm trên SynSE features hoặc `ca_concat_s1` (`gamma=1.0`)

In [ ]:
# !python {CMAP}/run_cr.py --feat synse --tasks 48 110 --seeds 1 --variants ca_concat_soft ca_lat_soft --out-root {OUT_DIR} --force